# 9 Expert Optical Prompt Step-by-Step

这个 notebook 用来逐步观察 9-expert 光学 prompt 的传播过程，而不是一次性保存大量图片。

目标流程：

1. 构造中心 `200 x 200` 手写数字输入。
2. 嵌入到带 padding 的大画布。
3. 用角谱法传播到 optical prompt 平面。
4. 分别测试两种 prompt：
   - **Prompt A**：全局透镜相位只在中心 `600 x 600` 真实 prompt 区域内生效；光栅相位按 9 个区域分区；kernel 随机相位置零。
   - **Prompt B**：透镜相位也按 9 个区域做局部透镜；光栅相位按 9 个区域分区；kernel 随机相位置零。
5. 传播到 9 个 expert aperture。
6. 经过若干层 identity expert propagation。
7. 观察 detector plane 和每个 expert 的能量分布。

注意：这里不训练、不加载 checkpoint，只看光场传播和 prompt 调制。

In [ ]:
%matplotlib inline

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.patches import Rectangle

# 自动寻找 opticalmoe/src，方便从仓库根目录、opticalmoe 目录或 notebooks 目录启动。
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / 'opticalmoe']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'src' / 'opticalmoe').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot find opticalmoe/src. Please run this notebook inside the repository.')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from opticalmoe.optics.angular_spectrum import AngularSpectrumPropagator

torch.set_grad_enabled(False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('device =', device)

## 1. 参数设置

这里采用 `3 x 3` experts，每个 expert 是 `200 x 200`。中间 expert bank 是 `600 x 600`，四周各加 `300 px` padding，所以总画布是 `1200 x 1200`。

如果你想测试不同传播距离，优先改下面几个变量：

- `INPUT_TO_PROMPT_CM`
- `PROMPT_TO_EXPERT_CM`
- `INTER_LAYER_CM`
- `FOCAL_LENGTH_CM`
- `GRATING_SIGN_X / GRATING_SIGN_Y`

In [ ]:
# 物理参数
WAVELENGTH_M = 532e-9
PIXEL_SIZE_M = 8e-6

# 几何参数
INPUT_SIZE = 200
EXPERT_SIZE = 200
BANK_SIZE = 3 * EXPERT_SIZE
PADDING = 300
CANVAS_SIZE = BANK_SIZE + 2 * PADDING
CANVAS_SHAPE = (CANVAS_SIZE, CANVAS_SIZE)
CANVAS_CENTER = (CANVAS_SIZE // 2, CANVAS_SIZE // 2)

# 3x3 experts 的中心。这里 expert bank 连续排布，不考虑 expert 间串扰。
CENTER_COORDS = [PADDING + EXPERT_SIZE // 2, PADDING + 3 * EXPERT_SIZE // 2, PADDING + 5 * EXPERT_SIZE // 2]
EXPERT_IDS = ['E00','E01','E02','E10','E11','E12','E20','E21','E22']

# 传播距离
INPUT_TO_PROMPT_CM = 20.0
PROMPT_TO_EXPERT_CM = 20.0
INTER_LAYER_CM = 5.0
NUM_IDENTITY_EXPERT_LAYERS = 5

# 如果 INPUT_TO_PROMPT_CM = PROMPT_TO_EXPERT_CM = 20 cm，则 1:1 成像薄透镜焦距是 10 cm。
FOCAL_LENGTH_CM = 10.0

# AngularSpectrumPropagator 的符号约定可能和直觉相反。
# 如果输出整体朝相反方向偏移，把 -1 改成 +1。
GRATING_SIGN_X = -1
GRATING_SIGN_Y = -1

# 选择本次重点观察的 amplitude case。
AMPLITUDE_CASE = 'uniform'  # uniform / center_only / onehot_E00 / onehot_E22 / diagonal

print('canvas:', CANVAS_SHAPE)
print('expert centers:', CENTER_COORDS)
print('distances cm:', INPUT_TO_PROMPT_CM, PROMPT_TO_EXPERT_CM, INTER_LAYER_CM)
print('focal length cm:', FOCAL_LENGTH_CM)

In [ ]:
def cm_to_m(value):
    return float(value) * 1e-2

def centered_slice(center, size):
    half = size // 2
    return slice(center - half, center + half)

def make_apertures(size):
    apertures = []
    for y in CENTER_COORDS:
        for x in CENTER_COORDS:
            apertures.append((centered_slice(y, size), centered_slice(x, size)))
    return apertures

EXPERT_APERTURES = make_apertures(EXPERT_SIZE)
PROMPT_REGIONS = make_apertures(EXPERT_SIZE)

def make_union_mask(apertures):
    mask = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    for ys, xs in apertures:
        mask[ys, xs] = 1.0
    return mask

EXPERT_UNION_MASK = make_union_mask(EXPERT_APERTURES)
PROMPT_UNION_MASK = make_union_mask(PROMPT_REGIONS)

def plot_layout():
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_facecolor('black')
    # input
    ys = centered_slice(CANVAS_CENTER[0], INPUT_SIZE)
    xs = centered_slice(CANVAS_CENTER[1], INPUT_SIZE)
    ax.add_patch(Rectangle((xs.start, ys.start), INPUT_SIZE, INPUT_SIZE, fill=False, edgecolor='orange', linewidth=2, label='input'))
    # prompt / expert regions
    for idx, (ys, xs) in enumerate(PROMPT_REGIONS):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor='cyan', linestyle='--', linewidth=1.2))
        ax.text(xs.start + 8, ys.start + 20, 'K' + EXPERT_IDS[idx][1:], color='cyan', fontsize=9)
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor='lime', linewidth=1.2))
        ax.text(xs.start + 8, ys.start + 42, EXPERT_IDS[idx], color='lime', fontsize=9)
    ax.axhline(CANVAS_CENTER[0], color='white', alpha=0.25)
    ax.axvline(CANVAS_CENTER[1], color='white', alpha=0.25)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(CANVAS_SIZE, 0)
    ax.set_title('canvas / input / 3x3 prompt regions / 3x3 experts')
    plt.show()

plot_layout()

## 2. 构造 200×200 手写数字输入

优先读取本地 `MNIST`。如果本地没有 MNIST，就用一个简单的合成 digit-like 图案兜底。无论来源如何，最终输入都会 resize 到 `200 x 200`，并嵌入画布中心。

In [ ]:
def synthetic_digit(size=200):
    img = torch.zeros(size, size, dtype=torch.float32)
    t = max(8, size // 13)
    m = max(12, size // 7)
    img[m:m+t, m:size-m] = 1.0
    img[m:size//2, m:m+t] = 1.0
    img[size//2-t//2:size//2+t//2, m:size-m] = 1.0
    img[size//2:size-m, size-m-t:size-m] = 1.0
    img[size-m-t:size-m, m:size-m] = 1.0
    img[int(size*0.68):int(size*0.76), int(size*0.35):int(size*0.52)] = 0.55
    return img

def load_mnist_or_synthetic(index=0, size=200):
    try:
        from torchvision.datasets import MNIST
        dataset = MNIST(root=str(PROJECT_ROOT / 'data'), train=False, download=False)
        img = dataset.data[index].float() / 255.0
        img = F.interpolate(img[None, None], size=(size, size), mode='bilinear', align_corners=False)[0, 0]
        print('Loaded MNIST sample, label =', int(dataset.targets[index]))
        return img
    except Exception as exc:
        print('MNIST not available, using synthetic digit. Reason:', repr(exc))
        return synthetic_digit(size)

input_img = load_mnist_or_synthetic(index=0, size=INPUT_SIZE).to(device)

input_field = torch.zeros((1, CANVAS_SIZE, CANVAS_SIZE), dtype=torch.complex64, device=device)
ys = centered_slice(CANVAS_CENTER[0], INPUT_SIZE)
xs = centered_slice(CANVAS_CENTER[1], INPUT_SIZE)
input_field[0, ys, xs] = input_img.to(torch.complex64)

plt.figure(figsize=(4, 4))
plt.imshow(input_img.detach().cpu(), cmap='gray')
plt.title('200x200 input amplitude')
plt.axis('off')
plt.show()

## 3. 可视化与传播工具函数

In [ ]:
def make_propagator(distance_cm):
    return AngularSpectrumPropagator(
        wavelength_m=WAVELENGTH_M,
        pixel_size_m=PIXEL_SIZE_M,
        grid_size=CANVAS_SHAPE,
        distance_m=cm_to_m(distance_cm),
    ).to(device)

prop_input_to_prompt = make_propagator(INPUT_TO_PROMPT_CM)
prop_prompt_to_expert = make_propagator(PROMPT_TO_EXPERT_CM)
prop_inter_layer = make_propagator(INTER_LAYER_CM)
print('Main pipeline propagation uses:', type(prop_input_to_prompt).__name__)
print('Input -> prompt, prompt -> expert, and identity expert propagation all use angular spectrum propagation.')

def intensity(field):
    if torch.is_complex(field):
        return torch.abs(field).square()
    return field.float()

def log_intensity(field):
    arr = intensity(field)
    if arr.ndim == 3:
        arr = arr[0]
    arr = arr.detach().cpu().numpy()
    return np.log10(arr / (arr.max() + 1e-12) + 1e-8)

def wrapped_phase(phase_or_field):
    if torch.is_complex(phase_or_field):
        phase = torch.angle(phase_or_field)
    else:
        phase = phase_or_field
    if phase.ndim == 3:
        phase = phase[0]
    return torch.remainder(phase, 2 * math.pi).detach().cpu().numpy()

def overlay_experts(ax, color='lime'):
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor=color, linewidth=0.8))
        ax.text(xs.start + 5, ys.start + 18, EXPERT_IDS[idx], color=color, fontsize=7)

def overlay_prompt_regions(ax, color='cyan'):
    for idx, (ys, xs) in enumerate(PROMPT_REGIONS):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor=color, linewidth=0.8, linestyle='--'))

def show_field(field, title='', overlay='experts', figsize=(6, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(log_intensity(field), cmap='inferno')
    if overlay == 'experts':
        overlay_experts(ax)
    elif overlay == 'prompt':
        overlay_prompt_regions(ax)
    ax.set_title(title)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(CANVAS_SIZE, 0)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label='log intensity')
    plt.show()

def show_phase(phase, title='', overlay='prompt'):
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(wrapped_phase(phase), cmap='twilight', vmin=0, vmax=2*math.pi)
    if overlay == 'prompt':
        overlay_prompt_regions(ax)
    elif overlay == 'experts':
        overlay_experts(ax)
    ax.set_title(title)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(CANVAS_SIZE, 0)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    plt.show()

def expert_energy(field):
    I = intensity(field)[0]
    values = []
    for ys, xs in EXPERT_APERTURES:
        values.append(float(I[ys, xs].sum().detach().cpu()))
    values = np.asarray(values, dtype=np.float64)
    return values / (values.sum() + 1e-12)

def show_energy_heatmap(values, title='expert energy ratio'):
    arr = np.asarray(values).reshape(3, 3)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(arr, cmap='magma')
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f'{arr[r,c]:.3f}', ha='center', va='center', color='white')
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    plt.show()

## 4. 传播到 optical prompt 平面

这一张图很关键：如果传播到 prompt 平面后，只有中心区域被照亮，那么后续无论怎么改分区振幅，非中心 expert 都不会有足够光。

In [ ]:
field_before_prompt = prop_input_to_prompt(input_field)

show_field(input_field, 'input plane intensity', overlay='prompt')
show_field(field_before_prompt, f'before prompt after {INPUT_TO_PROMPT_CM} cm propagation', overlay='prompt')

# 统计 prompt 9 个分区在 prompt 前实际拿到的光能。
prompt_incident = []
I_prompt = intensity(field_before_prompt)[0]
for ys, xs in PROMPT_REGIONS:
    prompt_incident.append(float(I_prompt[ys, xs].sum().detach().cpu()))
prompt_incident = np.asarray(prompt_incident)
prompt_incident_ratio = prompt_incident / (prompt_incident.sum() + 1e-12)
show_energy_heatmap(prompt_incident_ratio, 'incident energy on 9 prompt regions')
print('prompt incident ratios:', dict(zip(EXPERT_IDS, np.round(prompt_incident_ratio, 4))))

## 5. 构造两种 optical prompt

Prompt A：

```text
global lens phase: full canvas, not partitioned
kernel phase: zero phase, so exp(i*0)=1 in every region
grating phase: partitioned by 9 regions
```

Prompt B：

```text
local lens phase: each region has its own lens center
kernel phase: zero phase, so exp(i*0)=1 in every region
grating phase: partitioned by 9 regions
```

两个版本都支持 amplitude gate。

In [ ]:
yy = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[0]) * PIXEL_SIZE_M
xx = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[1]) * PIXEL_SIZE_M
Y_M, X_M = torch.meshgrid(yy, xx, indexing='ij')

def global_lens_phase():
    f = cm_to_m(FOCAL_LENGTH_CM)
    phase = -math.pi / (WAVELENGTH_M * f) * (X_M**2 + Y_M**2)
    # 外圈 padding 不是实际 prompt 区域，不应该显示或参与 lens phase。
    return phase * PROMPT_UNION_MASK

def local_lens_phase_map():
    f = cm_to_m(FOCAL_LENGTH_CM)
    phase = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    for ys, xs in PROMPT_REGIONS:
        cy = (ys.start + ys.stop) / 2
        cx = (xs.start + xs.stop) / 2
        y_local = (torch.arange(CANVAS_SIZE, device=device).float() - cy) * PIXEL_SIZE_M
        x_local = (torch.arange(CANVAS_SIZE, device=device).float() - cx) * PIXEL_SIZE_M
        YL, XL = torch.meshgrid(y_local, x_local, indexing='ij')
        phase[ys, xs] = (-math.pi / (WAVELENGTH_M * f) * (XL**2 + YL**2))[ys, xs]
    return phase

def zero_kernel_phase():
    # kernel 随机相位置零，相当于每个 kernel region 的额外调制为 exp(i*0)=1。
    # 这样观察现象时不会受到 random phase 干扰。
    return torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)

def grating_phase_map():
    phase = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    z = cm_to_m(PROMPT_TO_EXPERT_CM)
    for idx, (ys, xs) in enumerate(PROMPT_REGIONS):
        ey = (EXPERT_APERTURES[idx][0].start + EXPERT_APERTURES[idx][0].stop) / 2
        ex = (EXPERT_APERTURES[idx][1].start + EXPERT_APERTURES[idx][1].stop) / 2
        dy_m = (ey - CANVAS_CENTER[0]) * PIXEL_SIZE_M
        dx_m = (ex - CANVAS_CENTER[1]) * PIXEL_SIZE_M
        theta_y = math.atan(dy_m / z)
        theta_x = math.atan(dx_m / z)
        local = 2 * math.pi * (
            GRATING_SIGN_X * math.sin(theta_x) * X_M / WAVELENGTH_M
            + GRATING_SIGN_Y * math.sin(theta_y) * Y_M / WAVELENGTH_M
        )
        phase[ys, xs] = local[ys, xs]
    return phase

AMPLITUDE_CASES = {
    'uniform': [1,1,1,1,1,1,1,1,1],
    'center_only': [0,0,0,0,1,0,0,0,0],
    'onehot_E00': [1,0,0,0,0,0,0,0,0],
    'onehot_E22': [0,0,0,0,0,0,0,0,1],
    'diagonal': [1,0,0,0,1,0,0,0,1],
}

def amplitude_map(amplitudes):
    amap = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    for amp, (ys, xs) in zip(amplitudes, PROMPT_REGIONS):
        amap[ys, xs] = float(amp)
    return amap

KERNEL_PHASE = zero_kernel_phase()
GRATING_PHASE = grating_phase_map()
GLOBAL_LENS = global_lens_phase()
LOCAL_LENS = local_lens_phase_map()

def build_prompt(prompt_version='global_lens_partitioned_kernel', amplitude_case='uniform'):
    amplitudes = AMPLITUDE_CASES[amplitude_case]
    amap = amplitude_map(amplitudes)
    if prompt_version == 'global_lens_partitioned_kernel':
        total_phase = GLOBAL_LENS + KERNEL_PHASE + GRATING_PHASE
    elif prompt_version == 'local_lens_partitioned_kernel':
        total_phase = LOCAL_LENS + KERNEL_PHASE + GRATING_PHASE
    else:
        raise ValueError(prompt_version)
    transmission = amap * torch.exp(1j * total_phase).to(torch.complex64)
    return {
        'version': prompt_version,
        'amplitude_case': amplitude_case,
        'amplitudes': amplitudes,
        'amplitude_map': amap,
        'total_phase': total_phase,
        'transmission': transmission,
    }

In [ ]:
# 先看相位组件。
show_phase(GLOBAL_LENS, 'global lens phase, not partitioned')
show_phase(LOCAL_LENS, 'local lens phase, partitioned 3x3')
show_phase(KERNEL_PHASE, 'zero kernel phase, exp(i*0)=1')
show_phase(GRATING_PHASE, 'partitioned grating phase')

## 6. 跑完整传播流程

这里的专家层都是 identity，不训练任何相位，也没有随机 expert phase；等价于 expert phase 全部为 0、调制为 `exp(i*0)=1`。为了“不考虑串扰”，每次传播后都乘以 expert union aperture mask。

In [ ]:
def run_pipeline(prompt_version, amplitude_case='uniform'):
    prompt = build_prompt(prompt_version, amplitude_case)
    before_prompt = prop_input_to_prompt(input_field)
    after_prompt = before_prompt * prompt['transmission'].unsqueeze(0)
    expert_entrance = prop_prompt_to_expert(after_prompt)
    after_aperture = expert_entrance * EXPERT_UNION_MASK.unsqueeze(0).to(torch.complex64)
    layers = [after_aperture]
    field = after_aperture
    for _ in range(NUM_IDENTITY_EXPERT_LAYERS):
        field = prop_inter_layer(field)
        field = field * EXPERT_UNION_MASK.unsqueeze(0).to(torch.complex64)
        layers.append(field)
    return {
        'prompt': prompt,
        'before_prompt': before_prompt,
        'after_prompt': after_prompt,
        'expert_entrance': expert_entrance,
        'after_aperture': after_aperture,
        'layers': layers,
        'detector': layers[-1],
        'expert_energy': expert_energy(expert_entrance),
        'detector_energy': expert_energy(layers[-1]),
    }

def show_pipeline(result, title_prefix):
    prompt = result['prompt']
    print('prompt version:', prompt['version'])
    print('amplitude case:', prompt['amplitude_case'])
    print('amplitudes:', dict(zip(EXPERT_IDS, prompt['amplitudes'])))
    show_phase(prompt['total_phase'], title_prefix + ' total prompt phase')
    show_field(prompt['amplitude_map'], title_prefix + ' prompt amplitude map', overlay='prompt')
    show_field(result['before_prompt'], title_prefix + ' before prompt', overlay='prompt')
    show_field(result['after_prompt'], title_prefix + ' after prompt modulation', overlay='prompt')
    show_field(result['expert_entrance'], title_prefix + ' expert entrance', overlay='experts')
    show_energy_heatmap(result['expert_energy'], title_prefix + ' expert entrance energy')
    show_field(result['layers'][1], title_prefix + ' after identity propagation layer 1', overlay='experts')
    show_field(result['layers'][min(3, len(result['layers'])-1)], title_prefix + ' after identity propagation layer 3', overlay='experts')
    show_field(result['detector'], title_prefix + ' detector plane', overlay='experts')
    show_energy_heatmap(result['detector_energy'], title_prefix + ' detector energy')

## 7. Prompt A：中心 600×600 全局透镜 + 9 分区 grating + zero kernel phase

In [ ]:
result_A = run_pipeline('global_lens_partitioned_kernel', AMPLITUDE_CASE)
show_pipeline(result_A, 'Prompt A')

## 8. Prompt B：9 分区局部透镜 + 9 分区 grating + zero kernel phase

In [ ]:
result_B = run_pipeline('local_lens_partitioned_kernel', AMPLITUDE_CASE)
show_pipeline(result_B, 'Prompt B')

## Reference C: paper-style FFT convolution / global fan-out

This section reproduces the key structure of your previous code:

```text
Uout = ifft2(fft2(flip(input)) * fft2(P * lens))
P = sum_j amplitude_j * global_grating_j
```

This is different from Prompt A/B. Prompt A/B use angular spectrum propagation to a spatial prompt plane, then modulate only the light already present on that plane. Reference C is a convolution-theorem simulation with global gratings; every grating channel sees the whole input and can produce a shifted copy.

In [ ]:
def run_paper_convolution_reference(amplitude_case='uniform', use_center_600_aperture=True):
    amplitudes = AMPLITUDE_CASES[amplitude_case]
    yy = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[0]) * PIXEL_SIZE_M
    xx = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[1]) * PIXEL_SIZE_M
    Y, X = torch.meshgrid(yy, xx, indexing='ij')
    k = 2 * math.pi / WAVELENGTH_M
    f = cm_to_m(FOCAL_LENGTH_CM)
    d = cm_to_m(PROMPT_TO_EXPERT_CM)

    # Adjacent expert centers differ by 200 px = 1.6 mm.
    delta_target_m = EXPERT_SIZE * PIXEL_SIZE_M
    f1 = delta_target_m / (WAVELENGTH_M * d)

    orders = [(-1,-1),(0,-1),(1,-1),
              (-1, 0),(0, 0),(1, 0),
              (-1, 1),(0, 1),(1, 1)]
    P = torch.zeros(CANVAS_SHAPE, dtype=torch.complex64, device=device)
    for amp, (m, n) in zip(amplitudes, orders):
        G = torch.exp(1j * 2 * math.pi * (m * f1 * X + n * f1 * Y)).to(torch.complex64)
        P = P + float(amp) * G

    lens = torch.exp((-1j * k / (2 * f) * (X**2 + Y**2))).to(torch.complex64)
    Ltot = P * lens

    # Use this to respect the center 600x600 physical optical area; the outer area is padding.
    if use_center_600_aperture:
        Ltot = Ltot * PROMPT_UNION_MASK.to(torch.complex64)

    U0 = input_field[0]
    U0_flip = torch.flip(U0, dims=[0, 1])
    Uf = torch.fft.fft2(U0_flip)
    Pf = torch.fft.fft2(Ltot)
    Uout = torch.fft.fftshift(torch.fft.ifft2(Uf * Pf)).unsqueeze(0)

    return {
        'P': P,
        'Ltot': Ltot,
        'output': Uout,
        'expert_energy': expert_energy(Uout),
        'f1': f1,
        'delta_target_m': delta_target_m,
    }

ref = run_paper_convolution_reference(AMPLITUDE_CASE, use_center_600_aperture=True)
print('f1 =', ref['f1'])
print('target copy spacing mm =', ref['delta_target_m'] * 1e3)
show_phase(torch.angle(ref['P']), 'Reference C global fan-out grating phase sum', overlay='experts')
show_phase(torch.angle(ref['Ltot']), 'Reference C total convolution kernel phase', overlay='experts')
show_field(ref['output'], 'Reference C output intensity from FFT convolution', overlay='experts')
show_energy_heatmap(ref['expert_energy'], 'Reference C expert energy')

## 9. 振幅控制对比

下面固定一种 prompt 版本，快速比较不同 amplitude case 的 expert entrance 和 detector energy。

In [ ]:
PROMPT_VERSION_FOR_AMP_TEST = 'local_lens_partitioned_kernel'  # global_lens_partitioned_kernel / local_lens_partitioned_kernel
cases_to_test = ['uniform', 'center_only', 'onehot_E00', 'onehot_E22', 'diagonal']

for case in cases_to_test:
    res = run_pipeline(PROMPT_VERSION_FOR_AMP_TEST, case)
    print('\n===', PROMPT_VERSION_FOR_AMP_TEST, '/', case, '===')
    print('entrance energy:', dict(zip(EXPERT_IDS, np.round(res['expert_energy'], 3))))
    print('detector energy:', dict(zip(EXPERT_IDS, np.round(res['detector_energy'], 3))))
    show_energy_heatmap(res['expert_energy'], case + ' expert entrance energy')
    show_field(res['detector'], case + ' detector plane', overlay='experts', figsize=(5.5, 5.5))

## 10. 如何读这个 notebook 的结果

重点先看第 4 步的 `incident energy on 9 prompt regions`：

- 如果这里几乎只有中心 region 有能量，那么后续 amplitude 控制不会让角落专家变亮，因为那些区域本来就没有光。
- 如果 9 个 prompt regions 都有足够入射能量，再看 `expert entrance energy` 是否能跟随 amplitude case。
- 如果 `expert entrance` 正常，但 `detector plane` 漂移，说明 expert entrance 后仍有残余倾角，需要后续加 de-tilt / compensation。

Prompt A 和 Prompt B 的区别：

- Prompt A 更接近“中心 600×600 全局透镜 + 分区 grating + identity kernel”。
- Prompt B 更接近“每个 region 都有自己的局部透镜 + 分区 grating + identity kernel”。

如果你希望 200×200 输入被无一例外复制到 9 个 experts，需要确认 prompt 平面上 9 个 region 都能拿到一份有效输入光场；否则单纯分区 phase mask 只能调制已有光场，不能凭空给暗区域生成光。